In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(model_name_or_path="all-MiniLM-L6-v2")
v = model.encode("How does approximate nearest neighbor search work?")
v[0]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

np.float32(-0.020582063)

In [12]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()
documents = []
y = ""
for file in files:
    doc = file.parse()
    documents.append(doc)
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md":
        y = doc

y

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [19]:
vy = model.encode(y["content"])
vy.dot(v)

np.float32(0.45681345)

In [29]:
from gitsource import chunk_documents
from tqdm.auto import tqdm
import numpy as np
chunks = chunk_documents(documents,size =2000 ,step= 1000)
X = []
step = 50

contents = [x["content"] for x in chunks]

for i in tqdm(range(0,len(contents),step)):
    slice = contents[i:i + step]
    vc = model.encode(slice)
    X.extend(vc)

X = np.array(X)

scores = X.dot(v)

  0%|          | 0/6 [00:00<?, ?it/s]

In [33]:
idx = np.argmax(scores)
print(chunks[idx]["filename"])

02-vector-search/lessons/07-sqlitesearch-vector.md


In [37]:
from minsearch import VectorSearch
vindex= VectorSearch()
vindex.fit(X,chunks)

In [48]:
query ="What metric do we use to evaluate a search engine?"
res = vindex.search(model.encode(query),num_results=5)
res[0]["filename"]

'04-evaluation/lessons/01-intro.md'

In [50]:
from minsearch import Index

index = Index(text_fields=["content"])
index.fit(chunks)

res = index.search(query)
res[0]


{'start': 0,
 'content': '# Search\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=GYgpNKiuCJU&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\n## Search basics\n\nAt its core, every search engine does the same thing. It takes a query,\nscores every document for similarity, and returns the top results.\n\nFormally, there is a similarity function:\n\n```python\nscore = sim(query, document)\n```\n\nFor each document in the database, you compute this score. Then you\nrank all documents by score and return the top N. What makes a search\nengine different from another search engine is what `sim` actually\ncomputes.\n\n- text/lexical search (covered in this section): `sim` counts how\n  many words the query and the document share. It looks at the surface\n  form, the actual words, and matches them exactly.\n\n- vector/semantic search (covered in [module 2](../../02-vector-search/)):\n  `sim` compares the meaning of the query and the document. Same\n  function, different similarity m

In [63]:
query = "How do i store vectors in PostgreSQL"
resv = vindex.search(model.encode(query),num_results=5)
res = index.search(query,num_results=5)

In [64]:
for x in range(5):
    print(res[x]["filename"])
    print(resv[x]["filename"])


02-vector-search/lessons/02-embeddings.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/01-intro.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/01-intro.md
02-vector-search/lessons/08-pgvector.md


In [68]:
def rrf(result_lists,k=60,num_results = 5):
    scores = {}
    docs = {}

    for res in result_lists:
        for rank,doc in enumerate(res):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key,0) + 1/k+rank
            docs[key] = doc
    ranked = sorted(scores,key = scores.get,reverse=True)
    return [docs[key] for key in ranked[:num_results]]


In [69]:
results = rrf([res,resv])

In [70]:
results

[{'start': 1000,
  'content': 'dding model produces these vectors. It\'s a neural network\ntrained to capture meaning, so texts that mean similar things land on\nsimilar vectors. We measure how close two vectors are with a distance\nmetric. The most common one is cosine similarity.\n\nCosine similarity measures the angle between two vectors:\n\n- Vectors pointing in the same direction: similarity close to 1\n  (similar)\n- Vectors at right angles: similarity close to 0 (unrelated)\n- Vectors pointing in opposite directions: similarity close to -1\n  (opposite meaning)\n\nThe larger the cosine similarity, the more similar the two texts are\nin meaning.\n\n## Keyword search vs vector search\n\nHere\'s how the two approaches differ:\n\n- Keyword search matches exact words. Vector search matches meaning.\n- Keyword search suits specific terms, IDs, and names. Vector search\n  suits paraphrased questions and natural language.\n- Keyword search example: "pandas dataframe". Vector search exam